# Overpass extracts: shopping centres, local parks, beaches, doctors, clinics, hospitals

Queries OpenStreetMap once for `shop=mall`, `leisure=park`,
`natural=beach`, `amenity=doctors`, `amenity=clinic` and
`amenity=hospital` within Victoria, then splits, deduplicates by
location and exports each as its own file.

## Imports and configuration

In [1]:
import time

import numpy as np
import pandas as pd
import requests
from sklearn.neighbors import BallTree

OVERPASS_URL = "https://overpass-api.de/api/interpreter"
USER_AGENT = "vic-overpass-extracts-nikhil-gaba"

VIC_LAT_RANGE = (-39.3, -33.9)
VIC_LON_RANGE = (140.9, 150.1)
EARTH_RADIUS_M = 6_371_000
DEDUP_DISTANCE_M = 50

SHOPPING_CENTRES_PATH = "../../external_data/shopping_centres.csv"
LOCAL_PARKS_PATH = "../../external_data/parks_local_park.csv"
DOCTORS_PATH = "../../external_data/doctors_clinics.csv"
CLINICS_PATH = "../../external_data/specialised_clinics.csv"
HOSPITALS_PATH = "../../external_data/hospitals.csv"
BEACHES_PATH = "../../external_data/beaches.csv"

AREA_SETUP = '[out:json][timeout:300];area["ISO3166-2"="AU-VIC"]->.victoria;'
RETRY_WAIT_SECONDS = [15, 30, 60]

QUERIES = {
    "shop_mall": 'nwr["shop"="mall"](area.victoria);',
    "leisure_park": 'nwr["leisure"="park"](area.victoria);',
    "natural_beach": 'nwr["natural"="beach"](area.victoria);',
    "amenity_doctors": 'nwr["amenity"="doctors"](area.victoria);',
    "amenity_clinic": 'nwr["amenity"="clinic"](area.victoria);',
    "amenity_hospital": 'nwr["amenity"="hospital"](area.victoria);',
}

## Query Overpass, one tag at a time

Each tag is queried separately with retries on server errors (504
gateway timeout, 429 too many requests), since combining every tag into
one request risks timing out on the public Overpass instance. If a tag
still fails after every retry, its error is raised.

In [2]:
def query_overpass(query_body):
    """Run one Overpass query, retrying on server errors (504, 429)."""
    full_query = f"{AREA_SETUP}({query_body});out tags center;"
    for wait_seconds in RETRY_WAIT_SECONDS + [None]:
        response = requests.post(
            OVERPASS_URL,
            data={"data": full_query},
            headers={"User-Agent": USER_AGENT},
            timeout=300,
        )
        if response.status_code not in (429, 504):
            response.raise_for_status()
            return response.json()["elements"]
        if wait_seconds is None:
            response.raise_for_status()
        print(f"HTTP {response.status_code}, retrying in {wait_seconds} s")
        time.sleep(wait_seconds)


elements = []
for tag_name, query_body in QUERIES.items():
    tag_elements = query_overpass(query_body)
    print(f"{tag_name}: {len(tag_elements)} elements")
    elements.extend(tag_elements)

print(f"Total: {len(elements)} elements")

shop_mall: 383 elements
leisure_park: 14689 elements
HTTP 429, retrying in 15 s
HTTP 429, retrying in 30 s
natural_beach: 741 elements
amenity_doctors: 636 elements
HTTP 504, retrying in 15 s
HTTP 504, retrying in 30 s
amenity_clinic: 816 elements
HTTP 504, retrying in 15 s
HTTP 504, retrying in 30 s
HTTP 504, retrying in 60 s
amenity_hospital: 303 elements
Total: 17568 elements


## Flatten and split by tag

`element_to_row` is shared by every extract; the `id_prefix` argument
keeps each ID space distinct. Each element is routed to shopping
centres, local parks, beaches, doctors, clinics or hospitals based on
its own tags. An element matching none of the six tags (which shouldn't
happen, given the query) is counted and skipped rather than silently
dropped.

In [3]:
def element_to_row(element, id_prefix):
    """Flatten one Overpass element into an ID, name and location.

    Nodes carry their coordinates directly. Ways and relations carry a
    computed centre point instead, under the key "center".
    """
    location = element.get("center", element)
    return {
        "Facility ID": f"{id_prefix}_{element['type']}_{element['id']}",
        "name": element.get("tags", {}).get("name"),
        "Latitude": location["lat"],
        "Longitude": location["lon"],
    }


shopping_centre_rows = []
local_park_rows = []
doctors_rows = []
clinic_rows = []
hospital_rows = []
beach_rows = []
n_unmatched = 0

for element in elements:
    tags = element.get("tags", {})
    if tags.get("shop") == "mall":
        shopping_centre_rows.append(element_to_row(element, "MALL"))
    elif tags.get("leisure") == "park":
        local_park_rows.append(element_to_row(element, "OSM_PARK"))
    elif tags.get("amenity") == "doctors":
        doctors_rows.append(element_to_row(element, "DOCTORS"))
    elif tags.get("amenity") == "clinic":
        clinic_rows.append(element_to_row(element, "CLINIC"))
    elif tags.get("amenity") == "hospital":
        hospital_rows.append(element_to_row(element, "HOSPITAL"))
    elif tags.get("natural") == "beach":
        beach_rows.append(element_to_row(element, "BEACH"))
    else:
        n_unmatched += 1

shopping_centres = pd.DataFrame(shopping_centre_rows)
local_parks = pd.DataFrame(local_park_rows)
local_parks["PARK_CATEGORY"] = "local_park"
doctors = pd.DataFrame(doctors_rows)
clinics = pd.DataFrame(clinic_rows)
hospitals = pd.DataFrame(hospital_rows)
beaches = pd.DataFrame(beach_rows)
beaches["BEACH_CATEGORY"] = "beach"

print(f"Shopping centres: {len(shopping_centres)}")
print(f"Local parks: {len(local_parks)}")
print(f"Doctors: {len(doctors)}")
print(f"Clinics: {len(clinics)}")
print(f"Hospitals: {len(hospitals)}")
print(f"Beaches: {len(beaches)}")
print(f"Unmatched (unexpected): {n_unmatched}")

Shopping centres: 383
Local parks: 14689
Doctors: 636
Clinics: 816
Hospitals: 303
Beaches: 741
Unmatched (unexpected): 0


## Restrict to Victoria and deduplicate by location

Rows outside Victoria's bounding box are dropped first. Then, the same
real-world place mapped as both a node and a way produces two rows a
few metres apart, so duplicates are found by proximity rather than by
name: any pair within `DEDUP_DISTANCE_M` of each other is treated as
the same place, and only the first of each group is kept.

In [4]:
def restrict_to_victoria(df):
    """Drop rows whose coordinates fall outside Victoria's bounding box."""
    is_in_victoria = (
        df["Latitude"].between(*VIC_LAT_RANGE)
        & df["Longitude"].between(*VIC_LON_RANGE)
    )
    return df[is_in_victoria].reset_index(drop=True)


def deduplicate_by_location(df, label, distance_m=DEDUP_DISTANCE_M):
    """Drop rows whose coordinates fall within distance_m of an earlier
    row in the same table, treating them as the same real-world place.
    """
    if len(df) < 2:
        return df

    coordinates_rad = np.radians(df[["Latitude", "Longitude"]].to_numpy())
    tree = BallTree(coordinates_rad, metric="haversine")
    radius_rad = distance_m / EARTH_RADIUS_M

    is_duplicate = np.zeros(len(df), dtype=bool)
    for row in range(len(df)):
        if is_duplicate[row]:
            continue
        neighbours = tree.query_radius(
            coordinates_rad[[row]], r=radius_rad
        )[0]
        later_neighbours = neighbours[neighbours > row]
        is_duplicate[later_neighbours] = True

    n_dropped = is_duplicate.sum()
    print(f"{label}: dropping {n_dropped} of {len(df)} as duplicates")
    return df[~is_duplicate].reset_index(drop=True)


def clean_extract(df, label):
    """Restrict to Victoria, then deduplicate by location."""
    return deduplicate_by_location(restrict_to_victoria(df), label)


shopping_centres = clean_extract(shopping_centres, "Shopping centres")
local_parks = clean_extract(local_parks, "Local parks")
doctors = clean_extract(doctors, "Doctors")
clinics = clean_extract(clinics, "Clinics")
hospitals = clean_extract(hospitals, "Hospitals")
beaches = clean_extract(beaches, "Beaches")

Shopping centres: dropping 26 of 383 as duplicates
Local parks: dropping 254 of 14689 as duplicates
Doctors: dropping 70 of 636 as duplicates
Clinics: dropping 90 of 816 as duplicates
Hospitals: dropping 21 of 303 as duplicates
Beaches: dropping 14 of 741 as duplicates


## Export

In [5]:
shopping_centres.to_csv(SHOPPING_CENTRES_PATH, index=False)
local_parks.to_csv(LOCAL_PARKS_PATH, index=False)
doctors.to_csv(DOCTORS_PATH, index=False)
clinics.to_csv(CLINICS_PATH, index=False)
hospitals.to_csv(HOSPITALS_PATH, index=False)
beaches.to_csv(BEACHES_PATH, index=False)